# 01 · SELECT 基础与过滤（SQL 的“读数据”基本功）

> 如果说 SQL 是一门语言，`SELECT` 就是它的“句子”。这一课把“怎么把表里的数据按你想要的形状挑出来”讲透。
> 你在 Pandas 里写过 `df[df.age > 30][["name","age"]]`，SQL 做的是同一件事，只是写法顺序反过来了：**先选列，再过滤行，最后排序**。记住“你描述想要什么，数据库负责怎么拿”，这就是声明式。

## 本课地图
1. `SELECT ... FROM ...`：选列、起别名
2. `DISTINCT`：去重
3. `WHERE`：按行过滤（比较 / AND OR NOT / IN / BETWEEN / LIKE / IS NULL）
4. `ORDER BY`：排序；`LIMIT`：截取
5. `CASE WHEN`：行级分类（SQL 里的 if-else）
6. 预告：SQL 子句的真实执行顺序（02 课展开）

In [1]:
# 统一环境：连接数据库（绝对路径，可反复运行）
import sqlite3
import pandas as pd

# 两个库都用绝对路径写死，避免“相对路径在哪个工作目录下”的坑
SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
SS_DB  = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\superstore.db"

# 让 DataFrame 打印好看一点（你已经熟悉 Pandas，这里直接拿它当 SQL 的“显示层”）
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# 教学默认连 shop.db；需要 superstore 时再切
conn = sqlite3.connect(SHOP_DB)

def q(sql, conn=conn):
    """执行一条 SQL，把结果表直接返回成 Pandas DataFrame。"""
    return pd.read_sql_query(sql, conn)

print("已连接 shop.db。可用 q('SELECT ...') 跑查询。")
print("shop.db 表：", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

已连接 shop.db。可用 q('SELECT ...') 跑查询。
shop.db 表： ['customers', 'employees', 'order_items', 'orders', 'products']


## 1. SELECT ... FROM ...：选列与别名

最基本的形态：`SELECT 列 FROM 表`。`SELECT *` 是“全都要”，生产里少用（拉了一堆你不要的列，浪费 IO）。

In [2]:
# 只挑我关心的列，而不是 SELECT *
q('''
SELECT customer_id, name, region, age, segment
FROM customers
LIMIT 5
''')

,customer_id,name,region,age,segment
0,1,许军,华东,41,个人客户
1,2,马欣怡,华东,52,企业客户
2,3,郭秀英,华北,43,个人客户
3,4,梁艳,华南,51,家庭办公
4,5,何霞,华北,64,个人客户


**别名 `AS`**：给列或表起个临时名字。和 Pandas 里 `df.rename` 类似，但别名只存在于这一条查询的结果里。

In [3]:
# 列别名：把英文/难读的列改成好读的；也可以做计算列
q('''
SELECT  name        AS 客户姓名,
        age         AS 年龄,
        age * 12    AS 月龄        -- SELECT 里可以直接做算术
FROM customers
LIMIT 5
''')

,客户姓名,年龄,月龄
0,许军,41,492
1,马欣怡,52,624
2,郭秀英,43,516
3,梁艳,51,612
4,何霞,64,768


In [4]:
# 表别名：给表起短名（后面 JOIN 多表时你会爱死它）
# customers 缩写成 c，之后 c.name 就代表 customers.name
q('''
SELECT c.customer_id, c.name, c.city
FROM customers AS c
LIMIT 5
''')

,customer_id,name,city
0,1,许军,宁波
1,2,马欣怡,南京
2,3,郭秀英,北京
3,4,梁艳,广州
4,5,何霞,石家庄


> 细节：`AS` 其实可以省略（`customers c`），但写出来更清晰。习惯上我们**统一给表起短别名**，多表连接时一眼看出列来自哪张表。

## 2. DISTINCT：去重

`SELECT DISTINCT 列` 只保留“唯一的组合”。等价于 Pandas 的 `df[[...]].drop_duplicates()`。

In [5]:
# orders 表里出现过哪些地区？（不去重会有 2600 行重复）
q('''
SELECT DISTINCT region
FROM orders
ORDER BY region
''')

,region
0,华东
1,华中
2,华北
3,华南
4,西南


In [6]:
# DISTINCT 作用于“整行组合”，不是单列！
# 下面查的是 地区+城市 的唯一组合，不是单纯去重 region
q('''
SELECT DISTINCT region, city
FROM orders
ORDER BY region, city
LIMIT 10
''')

,region,city
0,华东,上海
1,华东,南京
2,华东,宁波
3,华东,杭州
4,华东,苏州
5,华中,南昌
6,华中,合肥
7,华中,武汉
8,华中,郑州
9,华中,长沙


> 易错点：`SELECT DISTINCT a, b` 去重的是 **(a,b) 这个组合**，不是只看 a。这是新手常踩的坑。另外 `COUNT(DISTINCT 列)` 是统计“有多少个不同值”，02 课会用。

## 3. WHERE：按行过滤

`WHERE` 在“选完列之后、聚合之前”逐行判断，只留条件为真的行。对应 Pandas 的布尔索引 `df[cond]`。

In [7]:
# 比较运算符：> >= < <= = <>（不等于）
# 找出 30 岁以上、华东地区的客户
q('''
SELECT customer_id, name, age, region
FROM customers
WHERE age >= 30 AND region = '华东'
LIMIT 8
''')

,customer_id,name,age,region
0,1,许军,41,华东
1,2,马欣怡,52,华东
2,20,宋伟,45,华东
3,23,赵一诺,65,华东
4,39,吴洋,52,华东
5,46,李子轩,32,华东
6,47,周佳琪,59,华东
7,51,梁一诺,30,华东


### AND / OR / NOT：逻辑组合

优先级是 `NOT > AND > OR`，跟数学里一样。**混用时务必用括号**，别靠脑子记优先级。

In [8]:
# 错误示范思路：想要“(华东或华北) 且 30岁以上”
# 如果不写括号，AND 会先结合，逻辑就错了
q('''
SELECT name, region, age
FROM customers
WHERE (region = '华东' OR region = '华北') AND age >= 30
LIMIT 8
''')

,name,region,age
0,许军,华东,41
1,马欣怡,华东,52
2,郭秀英,华北,43
3,何霞,华北,64
4,马芳,华北,64
5,徐娟,华北,52
6,宋伟,华东,45
7,赵一诺,华东,65


### IN：等于列表里任意一个

`x IN (a,b,c)` 等价于 `x=a OR x=b OR x=c`，写起来干净得多。

In [9]:
# 只看这几个地区的客户
q('''
SELECT name, region
FROM customers
WHERE region IN ('华东', '华北', '西南')
LIMIT 8
''')

,name,region
0,许军,华东
1,马欣怡,华东
2,郭秀英,华北
3,何霞,华北
4,郭杰,西南
5,马芳,华北
6,胡敏,西南
7,高佳琪,西南


### BETWEEN：闭区间

`x BETWEEN a AND b` 等价于 `a <= x <= b`，**两端都包含**。

In [10]:
# 年龄在 25 到 35 之间（含两端）
q('''
SELECT COUNT(*) AS 人数
FROM customers
WHERE age BETWEEN 25 AND 35
''')

,人数
0,63


### LIKE：字符串模糊匹配

`%` 匹配任意长度任意字符（包括 0 个），`_` 匹配**单个**字符。这是最简单的“正则”，别拿它当完整正则用。

In [11]:
# 名字里带“敏”的客户；以“王”开头的客户
q('''
SELECT customer_id, name
FROM customers
WHERE name LIKE '%敏%'
LIMIT 8
''')

,customer_id,name
0,12,胡敏
1,17,朱敏
2,43,郭敏
3,72,郭敏
4,103,朱敏
5,124,孙敏
6,165,刘敏
7,171,林敏


In [12]:
# 注意 LIKE 是逐字符匹配，% 的位置决定语义
# '张%' = 姓张；'%伟' = 名字最后一个字是伟；'林_' = 林 + 恰好一个字
print("姓张的客户数：",
      q("SELECT COUNT(*) c FROM customers WHERE name LIKE '张%'")["c"][0])
print("名字最后是“伟”的客户数：",
      q("SELECT COUNT(*) c FROM customers WHERE name LIKE '%伟'")["c"][0])

姓张的客户数： 12
名字最后是“伟”的客户数： 8


### IS NULL：判空（00 课强调过，这里再钉一遍）

`= NULL` 永远查不到东西，必须 `IS NULL` / `IS NOT NULL`。

In [13]:
# employees 里 manager_id 为空的，就是“大老板”那一级
q('''
SELECT employee_id, name, title, manager_id
FROM employees
WHERE manager_id IS NULL
''')

,employee_id,name,title,manager_id
0,1,张总,总经理,None


## 4. ORDER BY 排序 与 LIMIT 截取

`ORDER BY` 决定输出顺序，`DESC` 降序、`ASC` 升序（默认）。`LIMIT n` 只取前 n 行。

In [14]:
# 销售额最高的 10 条订单明细
q('''
SELECT order_id, product_id, quantity, discount, sales
FROM order_items
ORDER BY sales DESC
LIMIT 10
''')

,order_id,product_id,quantity,discount,sales
0,1,48,6,0.0,43202.52
1,181,48,6,0.0,43202.52
2,205,48,6,0.0,43202.52
3,212,48,6,0.0,43202.52
4,396,48,6,0.0,43202.52
5,469,48,6,0.0,43202.52
6,1048,48,6,0.0,43202.52
7,1059,48,6,0.0,43202.52
8,1880,48,6,0.0,43202.52
9,1098,48,6,0.1,38882.27


In [15]:
# 多列排序：先按地区升序，同地区再按年龄降序
q('''
SELECT name, region, age
FROM customers
ORDER BY region ASC, age DESC
LIMIT 10
''')

,name,region,age
0,赵一诺,华东,65
1,王平,华东,65
2,朱刚,华东,64
3,何芳,华东,64
4,孙桂英,华东,63
5,徐宇航,华东,61
6,周佳琪,华东,59
7,黄浩然,华东,59
8,宋娜,华东,57
9,朱静,华东,56


> 易错点：`ORDER BY` 排 NULL 的行为各数据库不同（SQLite 把 NULL 排最前）。另外**没有 ORDER BY 时，数据库不保证任何顺序**，别指望“跟表里存的顺序一样”。

## 5. CASE WHEN：行级 if-else

`CASE` 是 SQL 里唯一的“分支表达式”，常用来给行打标签、做分桶。它返回一个值，所以能出现在 SELECT 里。

In [16]:
# 给客户按年龄分桶：青年(<30) / 中年(30-50) / 老年(>50)
q('''
SELECT name, age,
  CASE
    WHEN age < 30 THEN '青年'
    WHEN age <= 50 THEN '中年'
    ELSE '老年'
  END AS age_group
FROM customers
LIMIT 10
''')

,name,age,age_group
0,许军,41,中年
1,马欣怡,52,老年
2,郭秀英,43,中年
3,梁艳,51,老年
4,何霞,64,老年
5,郭杰,51,老年
6,胡思远,27,青年
7,孙娜,54,老年
8,马芳,64,老年
9,何娜,53,老年


In [17]:
# CASE 配合判断：折扣 >= 0.4 算“大促单”，否则“常规单”
# 注意 CASE 是“从上往下命中第一个 WHEN”，写条件要注意顺序
q('''
SELECT order_id, discount, sales,
  CASE
    WHEN discount = 0            THEN '原价'
    WHEN discount < 0.3         THEN '小幅折扣'
    ELSE                             '大促折扣'
  END AS discount_band
FROM order_items
ORDER BY sales DESC
LIMIT 10
''')

,order_id,discount,sales,discount_band
0,1,0.0,43202.52,原价
1,181,0.0,43202.52,原价
2,205,0.0,43202.52,原价
3,212,0.0,43202.52,原价
4,396,0.0,43202.52,原价
5,469,0.0,43202.52,原价
6,1048,0.0,43202.52,原价
7,1059,0.0,43202.52,原价
8,1880,0.0,43202.52,原价
9,1098,0.1,38882.27,小幅折扣


> CASE 的两个坑：
> 1. **WHEN 是顺序匹配，命中第一个就停**，所以“从窄到宽”写条件，别反过来；
> 2. 忘了 `ELSE` 时，没命中任何分支的行该列会是 `NULL`，重要场景记得补 `ELSE`。

## 6. 预告：SQL 子句的真实执行顺序

你写 SQL 的顺序是 `SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY`，但数据库**执行**它们的顺序其实是：

```
FROM  →  WHERE  →  GROUP BY  →  HAVING  →  SELECT  →  ORDER BY  →  LIMIT
```

这解释了一个经典报错：**为什么 WHERE 里不能用 SELECT 里刚起的别名？** 因为 WHERE 执行时，SELECT 还没跑，别名还不存在。下一课讲聚合时你会再次撞到它（以及为什么 ORDER BY 里又能用别名）。

## 动手小练习（先自己写，再跑）

1. 查出 `customers` 表中所有“女性、年龄在 20 到 40 之间、segment 不是个人客户”的客户，按年龄升序取前 10。
2. 用 `DISTINCT` 查出 `orders` 表中所有出现过的 `ship_mode`。
3. 用 `CASE` 给 `order_items` 的 `profit` 打标签：正利润 `盈利`、利润为 0 或负 `亏损`。

> 参考答案在 notebook 项目与后续课程里会反复用到这些写法，这里先动手建立手感。

In [18]:
# 留作练习：在下面写你的 SQL，然后 q(...) 跑
# TODO(你): 练习1
# q(''' ... ''')

## 本课小结

- `SELECT 列 FROM 表` 选列，`AS` 起别名，表别名在多表时必备。
- `DISTINCT` 去重的是“整行组合”。
- `WHERE` 逐行过滤：比较 / AND OR NOT（善用括号）/ IN / BETWEEN（含两端）/ LIKE（% 任意、_ 单字符）/ IS NULL。
- `ORDER BY` 排序，`LIMIT` 截取；不排序就别假设顺序。
- `CASE WHEN ... THEN ... ELSE ... END` 是行级 if-else，顺序匹配、记得 ELSE。
- 写的顺序 ≠ 执行顺序，记住 `FROM→WHERE→GROUP→HAVING→SELECT→ORDER→LIMIT`。

## 来源与延伸

- SQLBolt 第 1~4 课（SELECT / 过滤 / 排序，交互式）：https://sqlbolt.com/
- Mode SQL Tutorial · Simple SELECT 语句：https://mode.com/sql-tutorial/sql-select-statement/
- SQLite 官方 SELECT 语法（含 WHERE / LIKE / CASE 细则）：https://www.sqlite.org/lang_select.html
- LeetCode 数据库入门题（用本课知识即可刷 Easy）：https://leetcode.com/problemset/database/
- DataLemur（面试向短题，适合碎片化练习）：https://datalemur.com/